# 🌍 ETL Températures Globales avec Spark

**Objectif** : Analyser l'évolution des températures mondiales par ville avec Apache Spark

**Dataset** : Global Land Temperatures by City (Kaggle) - 8M+ lignes, 1743-2013

## 📦 Étape 1 : Imports et configuration

On importe les bibliothèques nécessaires pour travailler avec Spark.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, min, max, year, month, round, to_date, desc, asc
import time

print("✅ Imports OK")

## 🚀 Étape 2 : Créer la session Spark

**Spark Session** = Point d'entrée pour utiliser Spark

**Configuration** :
- 4 GB de RAM pour le driver et les executors
- 8 partitions pour le parallélisme
- Spark UI disponible sur http://localhost:4040

In [ ]:
spark = SparkSession.builder \
    .appName("ETL_Temperatures") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")  # Moins de logs dans la console

print(f"✅ Spark {spark.version} démarré")
print(f"📊 Spark UI : http://localhost:4040")

## 📥 EXTRACT : Charger les données CSV

**Spark lit le CSV de manière distribuée** - Plus rapide que Pandas sur gros fichiers !

**Options** :
- `header=true` : Première ligne = noms de colonnes
- `inferSchema=true` : Spark devine automatiquement les types (int, string, etc.)

In [ ]:
print("📁 Chargement du fichier CSV...\n")
start = time.time()

# Charger le CSV
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("GlobalLandTemperaturesByCity.csv")

# Compter les lignes (déclenche le calcul)
nb_lignes = df.count()

print(f"✅ Chargement terminé en {time.time()-start:.1f}s")
print(f"📊 {nb_lignes:,} lignes chargées")
print(f"📊 {len(df.columns)} colonnes")
print(f"📊 {df.rdd.getNumPartitions()} partitions Spark\n")

# Afficher la structure
print("Structure des données :")
df.printSchema()

## 👀 Aperçu des données brutes

Regardons à quoi ressemblent les données avant nettoyage.

In [ ]:
print("Aperçu des 5 premières lignes :\n")
df.show(5, truncate=False)

# Compter les valeurs manquantes
print("\nValeurs manquantes par colonne :")
from pyspark.sql.functions import when, isnull

df.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df.columns
]).show()

## 🔧 TRANSFORM : Nettoyage et transformation

**Objectifs** :
1. Convertir la colonne `dt` (string) en vraie date
2. Extraire l'année et le mois
3. Renommer les colonnes pour plus de clarté
4. Supprimer les lignes avec température manquante

In [ ]:
print("🔧 Transformation des données...\n")

# 1. Convertir la date et extraire année/mois
df_clean = df \
    .withColumn("date", to_date(col("dt"), "yyyy-MM-dd")) \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", month(col("date")))

# 2. Renommer les colonnes pour plus de clarté
df_clean = df_clean.select(
    col("date"),
    col("year"),
    col("month"),
    col("AverageTemperature").alias("temperature"),
    col("AverageTemperatureUncertainty").alias("uncertainty"),
    col("City").alias("city"),
    col("Country").alias("country"),
    col("Latitude").alias("latitude"),
    col("Longitude").alias("longitude")
)

# 3. Supprimer les lignes avec valeurs manquantes importantes
avant = df_clean.count()
df_clean = df_clean.filter(
    col("temperature").isNotNull() & 
    col("city").isNotNull() & 
    col("country").isNotNull()
)
apres = df_clean.count()

print(f"✅ Transformation OK")
print(f"📊 Lignes avant : {avant:,}")
print(f"📊 Lignes après : {apres:,}")
print(f"📊 Supprimées : {avant-apres:,} ({(avant-apres)/avant*100:.1f}%)\n")

# Afficher le résultat
print("Données nettoyées :")
df_clean.show(10)

## 🔥 Analyse 1 : Top 10 des pays les plus chauds

**Question** : Quels sont les pays avec la température moyenne la plus élevée ?

**Méthode** : Grouper par pays, calculer la moyenne de température, trier

In [ ]:
print("🔥 Top 10 pays les plus chauds (température moyenne)\n")

pays_chauds = df_clean.groupBy("country") \
    .agg(
        round(avg("temperature"), 2).alias("temp_moyenne"),
        count("*").alias("nb_mesures")
    ) \
    .orderBy(desc("temp_moyenne")) \
    .limit(10)

pays_chauds.show(truncate=False)

## ❄️ Analyse 2 : Top 10 des pays les plus froids

**Question** : Quels sont les pays avec la température moyenne la plus basse ?

In [ ]:
print("❄️ Top 10 pays les plus froids (température moyenne)\n")

pays_froids = df_clean.groupBy("country") \
    .agg(
        round(avg("temperature"), 2).alias("temp_moyenne"),
        count("*").alias("nb_mesures")
    ) \
    .orderBy(asc("temp_moyenne")) \
    .limit(10)

pays_froids.show(truncate=False)

## 📈 Analyse 3 : Évolution de la température au fil du temps

**Question** : Comment la température globale a-t-elle évolué depuis 1750 ?

On regarde les 20 premières années et les 20 dernières.

In [ ]:
print("📈 Évolution de la température moyenne mondiale\n")

temp_par_annee = df_clean.groupBy("year") \
    .agg(
        round(avg("temperature"), 2).alias("temp_moyenne"),
        count("*").alias("nb_mesures")
    ) \
    .orderBy("year")

print("🕐 20 premières années :")
temp_par_annee.limit(20).show()

print("\n🕐 20 dernières années :")
temp_par_annee.orderBy(desc("year")).limit(20).show()

## 🌡️ Analyse 4 : Températures extrêmes

**Question** : Où et quand a-t-on enregistré les températures les plus extrêmes ?

On cherche les températures < -20°C ou > 40°C

In [ ]:
print("🌡️ Détection des températures extrêmes (< -20°C ou > 40°C)\n")

extremes = df_clean.filter(
    (col("temperature") < -20) | (col("temperature") > 40)
).select("date", "city", "country", "temperature")

nb_extremes = extremes.count()
print(f"📊 {nb_extremes:,} observations extrêmes trouvées\n")

print("🔥 Top 10 températures les PLUS ÉLEVÉES :")
extremes.orderBy(desc("temperature")).limit(10).show(truncate=False)

print("\n❄️ Top 10 températures les PLUS BASSES :")
extremes.orderBy(asc("temperature")).limit(10).show(truncate=False)

## 📅 Analyse 5 : Saisonnalité (température par mois)

**Question** : Quel est le mois le plus chaud/froid en moyenne ?

(Attention : hémisphère nord et sud inversés !)

In [ ]:
print("📅 Température moyenne mondiale par mois\n")

temp_par_mois = df_clean.groupBy("month") \
    .agg(
        round(avg("temperature"), 2).alias("temp_moyenne")
    ) \
    .orderBy("month")

temp_par_mois.show()

## 💾 LOAD : Sauvegarder les résultats

**Formats de sauvegarde** :
- **Parquet** : Format optimisé pour Spark, compressé, colonnaire
- **CSV** : Compatible Excel/pandas
- **JSON** : Pour les APIs

Les résultats seront dans le dossier `etl_output/`

In [ ]:
print("💾 Sauvegarde des résultats...\n")

# Créer le dossier de sortie
import os
os.makedirs("etl_output", exist_ok=True)

# 1. Sauvegarder toutes les données nettoyées (Parquet partitionné par année)
print("1️⃣ Sauvegarde données complètes (Parquet)...")
df_clean.write.mode("overwrite").partitionBy("year").parquet("etl_output/data_clean")
print("   ✅ OK\n")

# 2. Sauvegarder pays chauds (CSV)
print("2️⃣ Sauvegarde top pays chauds (CSV)...")
pays_chauds.coalesce(1).write.mode("overwrite").option("header", "true").csv("etl_output/pays_chauds")
print("   ✅ OK\n")

# 3. Sauvegarder évolution temporelle (CSV)
print("3️⃣ Sauvegarde évolution temporelle (CSV)...")
temp_par_annee.coalesce(1).write.mode("overwrite").option("header", "true").csv("etl_output/evolution_annuelle")
print("   ✅ OK\n")

# 4. Sauvegarder extrêmes (JSON)
print("4️⃣ Sauvegarde températures extrêmes (JSON)...")
extremes.write.mode("overwrite").json("etl_output/extremes")
print("   ✅ OK\n")

print("✅ Tous les résultats sont sauvegardés dans : etl_output/")

## 📊 Statistiques finales

Résumé de ce qu'on a fait.

In [ ]:
# Statistiques globales
stats = df_clean.select(
    avg("temperature").alias("temp_moyenne"),
    min("temperature").alias("temp_min"),
    max("temperature").alias("temp_max"),
    count("*").alias("nb_total")
).first()

print("="*60)
print("📊 RAPPORT FINAL - ETL TEMPÉRATURES GLOBALES")
print("="*60)
print(f"\n✅ Pipeline ETL terminé avec succès !\n")
print(f"📈 Lignes traitées : {stats['nb_total']:,}")
print(f"🌡️  Température moyenne globale : {stats['temp_moyenne']:.2f}°C")
print(f"❄️  Température minimale : {stats['temp_min']:.2f}°C")
print(f"🔥 Température maximale : {stats['temp_max']:.2f}°C")
print(f"⚠️  Observations extrêmes : {nb_extremes:,}")
print(f"\n💾 Résultats dans : etl_output/")
print(f"📊 Spark UI : http://localhost:4040")
print("="*60)

## 🛑 Fermeture (optionnel)

⚠️ **Ne pas exécuter** si vous voulez consulter Spark UI après !

Décommenter pour arrêter Spark.

In [ ]:
# spark.stop()
# print("✅ Session Spark arrêtée")

print("ℹ️  Session Spark toujours active")
print("ℹ️  Spark UI reste disponible sur http://localhost:4040")